[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20II%20-%20The%20End-to-End%20Supply%20Chain%20%28Problems%2047-101%29/A.%20Foundational%20Analytics%20%26%20Inventory%20Control%20%28The%20Language%20of%20Supply%20Chain%29/047.%20The%20Demand%20Forecasting%20-%20Moving%20Average%20%26%20Weighted%20MA%20Problem/P47-Tier-3_executed.ipynb)

---



# 47. The Demand Forecasting Problem
## Tier 3: Particle Swarm Optimization for Weighted Moving Average Weights

### Goal

Use Particle Swarm Optimization (PSO) to search for nonnegative weighted moving average (WMA) weights that sum to one and minimize mean absolute deviation (MAD) on historical rolling forecasts. The notebook evaluates 3-period and 4-period windows, compares the resulting errors with simple and linearly weighted moving averages, and uses the selected weights to forecast the next period.

PSO is a stochastic search procedure. It returns the best candidate found during a run; it does not certify that this candidate is a global optimum.

### Inputs, outputs, and scope

- **Input:** weekly demand for 12 periods and the PSO settings shown in the code.
- **Decision variables:** one weight per period in the moving-average window.
- **Output:** the best-found weights, their rolling-backtest MAD, a next-period forecast, and diagnostic histories for fitness and swarm diversity.
- **Scope:** this example uses a single historical series and optimizes and evaluates weights on that same series. It is an in-sample rolling-origin backtest, not an independent test-set evaluation. It does not model seasonality, trends, changing demand regimes, forecast uncertainty, or inventory and service-level costs.

### Modeling assumptions

Each forecast is a weighted average of the most recent observations available before its target period. Weights are nonnegative and sum to one, so each forecast is a convex combination of those observations. The weight vector is held fixed across the rolling-backtest periods for a given window size. The first four zero-based target positions are omitted so that the two window sizes are assessed on the same target periods.

### Solution sequence

1. Define the PSO parameters and initialize particles with feasible weight vectors.
2. Evaluate each particle by calculating rolling-origin WMA forecasts and their MAD.
3. Update each particle's velocity using inertia, its personal best, and the swarm's global best.
4. Update particle positions and enforce the weight constraints.
5. Repeat the evaluation and updates while recording best fitness and swarm diversity.
6. Run the search separately for each window size, compare the results with fixed-weight baselines, and inspect sensitivity to PSO settings.

### Interpreting the results

A lower MAD indicates smaller average absolute errors on the evaluated historical targets. The PSO result is the best candidate encountered in that stochastic run, not a proof of global optimality or future accuracy. Similar results across parameter settings provide evidence about run-to-run sensitivity, but do not establish robustness to changes in the demand process. Because the same series is used to choose and assess the weights, an independent evaluation would be needed to estimate out-of-sample performance.


## Particle Swarm Optimization and the WMA Objective

### Weighted moving average and rolling MAD

For a window of size $k$, the weight vector assigns one weight to each observation in the window, ordered from oldest to most recent. If the forecast target has zero-based index $t$, the available history is $D_{t-k},\ldots,D_{t-1}$. The forecast is

$$
F_t(\mathbf{w}) = \sum_{j=1}^{k} w_j D_{t-k+j-1}.
$$

The weights satisfy

$$
\sum_{j=1}^{k} w_j = 1, \qquad w_j \geq 0 \quad (j=1,\ldots,k).
$$

The fitness is the mean absolute deviation over the common set of rolling forecast targets:

$$
\operatorname{MAD}(\mathbf{w}; k) = \frac{1}{|T|}\sum_{t\in T}\left|D_t-F_t(\mathbf{w})\right|,
\qquad T=\{4,5,\ldots,N-1\}.
$$

Here, $D_t$ is observed demand, $F_t$ is the forecast made using only observations before period $t$, $N$ is the number of observations, and $|T|$ is the number of evaluated targets. The code uses `calculate_forecast_error` to calculate this objective. A zero-based target index of 4 means that the fifth observation is the first one evaluated; both window sizes use these same target periods. This alignment makes their MAD values directly comparable.

For the next period, whose index is $N$, the forecast uses the latest $k$ observations and the selected weight vector. Historical MAD measures fit on the rolling targets; it is not an independent estimate of future error.

### Particle state and update equations

A particle represents one candidate weight vector. Its position is the candidate solution, and its velocity determines how that candidate moves through the weight space. Each particle remembers its best position, while the swarm shares the best position found by any particle. These memories guide the search: the personal best encourages individual learning, and the global best shares information across the swarm.

For particle $i$ at iteration $t$, PSO updates velocity and then position as follows:

$$
v_i^{t+1} = \omega v_i^t + c_1 r_1 \left(p_i^t-x_i^t\right) + c_2 r_2 \left(g^t-x_i^t\right),
$$

$$
x_i^{t+1}=x_i^t+v_i^{t+1}.
$$

Here, $x_i^t$ is the particle's current position, $v_i^t$ is its velocity, $p_i^t$ is its personal best position, and $g^t$ is the swarm's global best position. The inertia coefficient $\omega$ controls how much of the previous velocity is retained. The cognitive and social coefficients $c_1$ and $c_2$ control attraction toward the personal and global bests. The random vectors $r_1$ and $r_2$ have components drawn uniformly from $[0,1]$. In the code, `Particle.update_velocity` applies the velocity equation and limits its Euclidean magnitude; `Particle.update_position` moves the particle and enforces feasible weights by clipping negative components to zero and normalizing the remaining components to sum to one.

### Search procedure and computational cost

The algorithm initializes a swarm, evaluates every particle, updates personal and global bests, and then updates velocities and positions. It records the best MAD and the average pairwise distance between particle positions at each iteration. This distance is a descriptive measure of swarm diversity, not a measure of forecast accuracy.

With swarm size $S$, $I$ iterations, $|T|$ backtest targets, and window size $k$, fitness evaluation costs $O(SI|T|k)$. The pairwise diversity calculation adds $O(IS^2k)$ work. PSO is stochastic and does not provide a general guarantee that it will find the global minimum. The stopping rule in this implementation checks whether the best MAD has changed by less than the configured threshold over a recent interval; this is a practical stopping criterion, not an optimality certificate.


In [ ]:
import warnings
from dataclasses import dataclass
from typing import Dict, List, Optional

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

warnings.filterwarnings("ignore")

plt.style.use("seaborn-v0_8")
sns.set_palette("husl")

print("Libraries imported successfully for PSO optimization analysis.")

In [ ]:
@dataclass
class PSOParameters:
    """Configuration values for the Particle Swarm Optimization search."""

    swarm_size: int = 20
    max_iterations: int = 100
    inertia_weight: float = 0.7
    cognitive_coeff: float = 1.5
    social_coeff: float = 1.5
    velocity_limit: float = 0.5
    convergence_threshold: float = 1e-6


class Particle:
    """One candidate weight vector and its movement through the search space."""

    def __init__(self, dimension: int):
        """Create a feasible random position and a small random velocity."""
        self.dimension = dimension
        self.position = self._initialize_position()
        self.velocity = self._initialize_velocity()
        self.personal_best_position = self.position.copy()
        self.personal_best_fitness = float("inf")
        self.fitness_history = []

    def _initialize_position(self) -> np.ndarray:
        """Draw nonnegative weights and normalize them to sum to one."""
        random_weights = np.random.random(self.dimension)
        return random_weights / np.sum(random_weights)

    def _initialize_velocity(self) -> np.ndarray:
        """Initialize movement in each weight dimension."""
        return np.random.uniform(-0.1, 0.1, self.dimension)

    def update_velocity(
        self,
        global_best_position: np.ndarray,
        params: PSOParameters,
    ) -> None:
        """Apply the PSO velocity equation and cap its Euclidean magnitude."""
        r1 = np.random.random(self.dimension)
        r2 = np.random.random(self.dimension)

        cognitive_component = (
            params.cognitive_coeff
            * r1
            * (self.personal_best_position - self.position)
        )
        social_component = (
            params.social_coeff
            * r2
            * (global_best_position - self.position)
        )

        self.velocity = (
            params.inertia_weight * self.velocity
            + cognitive_component
            + social_component
        )

        velocity_magnitude = np.linalg.norm(self.velocity)
        if velocity_magnitude > params.velocity_limit:
            self.velocity = (
                self.velocity / velocity_magnitude
            ) * params.velocity_limit

    def update_position(self) -> None:
        """Move the particle and project its position onto feasible weights."""
        self.position = self.position + self.velocity
        self.position = np.maximum(self.position, 0)

        weight_sum = np.sum(self.position)
        if weight_sum > 0:
            self.position = self.position / weight_sum
        else:
            # A zero vector cannot be normalized, so draw a new feasible position.
            self.position = self._initialize_position()


print("PSO classes defined successfully.")

In [ ]:
class PSOWeightOptimizer:
    """Search for WMA weights that minimize rolling-origin forecast MAD."""

    def __init__(
        self,
        demand_data: List[float],
        params: Optional[PSOParameters] = None,
    ):
        """Store the demand series and PSO settings used by this optimizer."""
        self.demand_data = np.asarray(demand_data, dtype=float)
        if self.demand_data.ndim != 1 or len(self.demand_data) == 0:
            raise ValueError("demand_data must be a nonempty one-dimensional series.")

        self.params = params or PSOParameters()
        self.optimization_history = {}

    def calculate_forecast_error(
        self,
        weights: np.ndarray,
        window_size: int,
        start_period: int = 4,
    ) -> float:
        """Calculate MAD on targets forecast using only earlier observations.

        start_period is the zero-based index of the first target to evaluate.
        """
        if window_size <= 0 or window_size > len(self.demand_data):
            raise ValueError("window_size must be positive and no larger than the data.")
        if len(weights) != window_size:
            raise ValueError("The number of weights must equal window_size.")
        if start_period < 0:
            raise ValueError("start_period must be a nonnegative target index.")

        first_target = max(start_period, window_size)
        if first_target >= len(self.demand_data):
            return float("inf")

        errors = []
        for target_index in range(first_target, len(self.demand_data)):
            history = self.demand_data[
                target_index - window_size:target_index
            ]
            forecast = np.dot(weights, history)
            actual = self.demand_data[target_index]
            errors.append(abs(actual - forecast))

        return float(np.mean(errors)) if errors else float("inf")

    def optimize_weights(self, window_size: int) -> Dict:
        """Run PSO for one window size and return its best-found candidate."""
        if window_size <= 0 or window_size >= len(self.demand_data):
            raise ValueError("window_size must be positive and smaller than the data length.")

        swarm = [
            Particle(window_size)
            for _ in range(self.params.swarm_size)
        ]
        if not swarm:
            raise ValueError("swarm_size must be at least one.")

        global_best_position = swarm[0].position.copy()
        global_best_fitness = float("inf")
        fitness_history = []
        diversity_history = []

        print(f"--- Optimizing {window_size}-period WMA ---")

        for iteration in range(self.params.max_iterations):
            for particle in swarm:
                fitness = self.calculate_forecast_error(
                    particle.position,
                    window_size,
                )
                particle.fitness_history.append(fitness)

                if fitness < particle.personal_best_fitness:
                    particle.personal_best_fitness = fitness
                    particle.personal_best_position = particle.position.copy()

                if fitness < global_best_fitness:
                    global_best_fitness = fitness
                    global_best_position = particle.position.copy()

            fitness_history.append(global_best_fitness)

            positions = np.array([particle.position for particle in swarm])
            pairwise_distances = [
                np.linalg.norm(positions[i] - positions[j])
                for i in range(len(swarm))
                for j in range(i + 1, len(swarm))
            ]
            diversity = float(np.mean(pairwise_distances)) if pairwise_distances else 0.0
            diversity_history.append(diversity)

            if iteration % 20 == 0:
                print(f"Iteration {iteration}: Best MAD = {global_best_fitness:.3f}")

            for particle in swarm:
                particle.update_velocity(global_best_position, self.params)
                particle.update_position()

            if iteration > 10 and len(fitness_history) > 10:
                recent_improvement = abs(
                    fitness_history[-1] - fitness_history[-10]
                )
                if recent_improvement < self.params.convergence_threshold:
                    print(f"Stopping criterion reached at iteration {iteration}.")
                    break

        final_weights = global_best_position.copy()
        next_period_forecast = float(
            np.dot(final_weights, self.demand_data[-window_size:])
        )

        return {
            "best_weights": final_weights,
            "best_mad": global_best_fitness,
            "next_period_forecast": next_period_forecast,
            "fitness_history": fitness_history,
            "diversity_history": diversity_history,
            "window_size": window_size,
            "iterations_completed": iteration + 1,
        }

    def optimize_multiple_windows(self, window_sizes: List[int]) -> Dict:
        """Optimize each requested window size and select the lowest-MAD result."""
        results = {}

        print("=== PSO-Optimized Weighted Moving Average ===")
        print()

        for window_size in window_sizes:
            if window_size >= len(self.demand_data):
                print(f"Skipping window size {window_size} (too large for data).")
                continue

            result = self.optimize_weights(window_size)
            results[window_size] = result

            print(
                "Best weights: "
                f"{[f'{weight:.4f}' for weight in result['best_weights']]}"
            )
            print(
                "Next-period forecast: "
                f"{result['next_period_forecast']:.1f} units"
            )
            print()

        if not results:
            raise ValueError("No valid window sizes were provided.")

        best_window = min(
            results,
            key=lambda size: results[size]["best_mad"],
        )
        best_result = results[best_window]

        print(f"Best found configuration: {best_window}-period WMA")
        print(f"Rolling-backtest MAD: {best_result['best_mad']:.3f}")
        print(
            "Next-period forecast: "
            f"{best_result['next_period_forecast']:.1f} units"
        )

        return {
            "all_results": results,
            "best_window_size": best_window,
            "best_result": best_result,
        }


print("PSOWeightOptimizer class defined successfully.")

In [ ]:
# Weekly wireless-headphone demand used throughout the example.
demand_data = [450, 380, 520, 610, 485, 390, 475, 550, 425, 480, 515, 460]

pso_params = PSOParameters(
    swarm_size=20,
    max_iterations=100,
    inertia_weight=0.7,
    cognitive_coeff=1.5,
    social_coeff=1.5,
    velocity_limit=0.5,
)

optimizer = PSOWeightOptimizer(demand_data, pso_params)

print("=== PSO-Optimized Weighted Moving Average ===")
print(f"Demand data: {demand_data}")
print(f"Data periods: {len(demand_data)}")
print(
    f"PSO parameters: swarm={pso_params.swarm_size}, "
    f"maximum iterations={pso_params.max_iterations}"
)
print()

In [ ]:
# Compare the same rolling-backtest target periods for both window sizes.
window_sizes = [3, 4]
optimization_results = optimizer.optimize_multiple_windows(window_sizes)

all_results = optimization_results["all_results"]
best_window_size = optimization_results["best_window_size"]
best_result = optimization_results["best_result"]

In [ ]:
print("=== Detailed PSO Analysis ===")
print()


def calculate_traditional_performance(demand_data, window_size):
    """Calculate rolling MAD for simple and linearly weighted moving averages."""
    demand = np.asarray(demand_data, dtype=float)
    first_target = max(4, window_size)
    if first_target >= len(demand):
        raise ValueError("The data must contain at least one eligible forecast target.")

    sma_errors = []
    for target_index in range(first_target, len(demand)):
        recent_data = demand[target_index - window_size:target_index]
        forecast = np.mean(recent_data)
        actual = demand[target_index]
        sma_errors.append(abs(actual - forecast))

    # In this oldest-to-newest ordering, larger weights go to more recent periods.
    traditional_weights = np.linspace(0.5, 1.5, window_size)
    traditional_weights = traditional_weights / np.sum(traditional_weights)

    wma_errors = []
    for target_index in range(first_target, len(demand)):
        recent_data = demand[target_index - window_size:target_index]
        forecast = np.dot(traditional_weights, recent_data)
        actual = demand[target_index]
        wma_errors.append(abs(actual - forecast))

    return {
        "SMA_MAD": float(np.mean(sma_errors)),
        "WMA_MAD": float(np.mean(wma_errors)),
        "Traditional_Weights": traditional_weights,
    }


print("Performance comparison on common rolling-backtest targets:")
print(
    "Window | SMA MAD | Traditional WMA MAD | PSO WMA MAD "
    "| Improvement | PSO weights"
)
print("-" * 100)

for window_size in window_sizes:
    if window_size in all_results:
        traditional = calculate_traditional_performance(
            demand_data,
            window_size,
        )
        pso_result = all_results[window_size]

        improvement = (
            (traditional["WMA_MAD"] - pso_result["best_mad"])
            / traditional["WMA_MAD"]
            * 100
        )
        weights_str = ", ".join(
            f"{weight:.4f}" for weight in pso_result["best_weights"]
        )

        print(
            f"{window_size:6d} | "
            f"{traditional['SMA_MAD']:7.2f} | "
            f"{traditional['WMA_MAD']:19.2f} | "
            f"{pso_result['best_mad']:11.3f} | "
            f"{improvement:10.1f}% | [{weights_str}]"
        )

print()
print(f"Best found configuration: {best_window_size}-period WMA")
print(f"Rolling-backtest MAD: {best_result['best_mad']:.3f}")
print(
    "Best-found weights: "
    f"{[f'{weight:.4f}' for weight in best_result['best_weights']]}"
)
print(
    f"Next-period forecast: "
    f"{best_result['next_period_forecast']:.1f} units"
)

In [ ]:
print("=== Convergence Analysis ===")

best_window_result = all_results[best_window_size]
fitness_history = best_window_result["fitness_history"]
diversity_history = best_window_result["diversity_history"]

print(f"Analysis for {best_window_size}-period WMA:")
print(f"Initial best MAD: {fitness_history[0]:.3f}")
print(f"Final best MAD: {fitness_history[-1]:.3f}")
if fitness_history[0] != 0:
    improvement_percent = (
        (fitness_history[0] - fitness_history[-1])
        / fitness_history[0]
        * 100
    )
    print(f"Change in best MAD: {improvement_percent:.1f}%")
else:
    print("The initial best MAD is zero, so a percentage change is undefined.")
print(f"Iterations completed: {best_window_result['iterations_completed']}")
print(f"Initial swarm diversity: {diversity_history[0]:.4f}")
print(f"Final swarm diversity: {diversity_history[-1]:.4f}")

# This diagnostic uses a relative change over ten recorded iterations.
convergence_point = None
for i in range(10, len(fitness_history)):
    reference_fitness = fitness_history[i - 10]
    if reference_fitness == 0:
        relative_improvement = 0.0
    else:
        relative_improvement = abs(
            fitness_history[i] - reference_fitness
        ) / abs(reference_fitness)

    if relative_improvement < 0.01:
        convergence_point = i
        break

if convergence_point is not None:
    print(
        "A change below 1% over the diagnostic interval was first "
        f"observed at recorded iteration {convergence_point}."
    )
else:
    print("The diagnostic threshold was not reached in the recorded history.")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle(
    "PSO Weight Optimization for Demand Forecasting",
    fontsize=16,
    fontweight="bold",
)

# Compare the best-so-far MAD histories for the tested window sizes.
ax1 = axes[0, 0]
colors = ["blue", "red", "green"]
for i, (window_size, result) in enumerate(all_results.items()):
    iterations = range(len(result["fitness_history"]))
    ax1.plot(
        iterations,
        result["fitness_history"],
        linewidth=2,
        label=f"{window_size}-period WMA",
        color=colors[i % len(colors)],
    )

ax1.set_xlabel("Recorded iteration")
ax1.set_ylabel("Best-so-far MAD (demand units)")
ax1.set_title("PSO Best-Fitness History")
ax1.legend()
ax1.grid(True, alpha=0.3)
if all(
    value > 0
    for result in all_results.values()
    for value in result["fitness_history"]
):
    ax1.set_yscale("log")

# Compare methods on the same target periods for each window size.
ax2 = axes[0, 1]
methods = []
mads = []
bar_colors = []

for window_size in window_sizes:
    if window_size in all_results:
        traditional = calculate_traditional_performance(
            demand_data,
            window_size,
        )
        pso_result = all_results[window_size]

        methods.extend(
            [
                f"SMA ({window_size})",
                f"Linear WMA ({window_size})",
                f"PSO WMA ({window_size})",
            ]
        )
        mads.extend(
            [
                traditional["SMA_MAD"],
                traditional["WMA_MAD"],
                pso_result["best_mad"],
            ]
        )
        bar_colors.extend(["lightblue", "lightcoral", "gold"])

bars = ax2.bar(range(len(methods)), mads, color=bar_colors, alpha=0.7)
ax2.set_xlabel("Forecasting method and window size")
ax2.set_ylabel("Rolling-backtest MAD (demand units)")
ax2.set_title("Forecast Error Comparison")
ax2.set_xticks(range(len(methods)))
ax2.set_xticklabels(methods, rotation=45, ha="right")
ax2.grid(True, alpha=0.3, axis="y")

for i, method in enumerate(methods):
    if "PSO" in method:
        bars[i].set_edgecolor("black")
        bars[i].set_linewidth(2)

# Weight index zero corresponds to the oldest observation in the window.
ax3 = axes[1, 0]
best_weights = best_result["best_weights"]
labels = [
    f"t-{len(best_weights) - i}"
    for i in range(len(best_weights))
]
traditional_weights = calculate_traditional_performance(
    demand_data,
    best_window_size,
)["Traditional_Weights"]

x = np.arange(len(labels))
width = 0.35
ax3.bar(
    x - width / 2,
    traditional_weights,
    width,
    label="Linear WMA",
    color="lightcoral",
    alpha=0.7,
)
ax3.bar(
    x + width / 2,
    best_weights,
    width,
    label="PSO best-found weights",
    color="gold",
    alpha=0.7,
)
ax3.set_xlabel("Period relative to the forecast target")
ax3.set_ylabel("Weight")
ax3.set_title(f"Weight Comparison ({best_window_size}-period window)")
ax3.set_xticks(x)
ax3.set_xticklabels(labels)
ax3.legend()
ax3.grid(True, alpha=0.3, axis="y")

# Diversity describes spread among candidate positions, not solution quality.
ax4 = axes[1, 1]
iterations = range(len(diversity_history))
ax4.plot(iterations, diversity_history, linewidth=2, color="purple")
ax4.set_xlabel("Recorded iteration")
ax4.set_ylabel("Mean pairwise distance between weight vectors")
ax4.set_title("Swarm Diversity")
ax4.grid(True, alpha=0.3)

if convergence_point is not None:
    ax4.axvline(
        x=convergence_point,
        color="red",
        linestyle="--",
        alpha=0.7,
        label=f"Diagnostic threshold at {convergence_point}",
    )
    ax4.legend()

plt.tight_layout()
plt.show()

print("PSO diagnostic visualizations are ready.")

In [ ]:
print("=== Sensitivity Analysis ===")

# Each run is stochastic; these settings explore how selected PSO parameters
# affect the best candidate found for the selected window size.
parameter_tests = [
    {"name": "Default", "params": PSOParameters()},
    {"name": "High Inertia", "params": PSOParameters(inertia_weight=0.9)},
    {"name": "Low Inertia", "params": PSOParameters(inertia_weight=0.4)},
    {"name": "Large Swarm", "params": PSOParameters(swarm_size=30)},
    {"name": "Small Swarm", "params": PSOParameters(swarm_size=10)},
]

sensitivity_results = []
for test in parameter_tests:
    test_optimizer = PSOWeightOptimizer(demand_data, test["params"])
    result = test_optimizer.optimize_weights(best_window_size)

    sensitivity_results.append(
        {
            "Test_Name": test["name"],
            "Best_MAD": result["best_mad"],
            "Iterations": result["iterations_completed"],
            "Weights": result["best_weights"],
        }
    )

print("\nParameter sensitivity results:")
print("Test name        | Best MAD | Iterations | First three weights")
print("-" * 70)

for result in sensitivity_results:
    weights_pattern = ", ".join(
        f"{weight:.3f}" for weight in result["Weights"][:3]
    )
    print(
        f"{result['Test_Name']:15s} | "
        f"{result['Best_MAD']:8.3f} | "
        f"{result['Iterations']:10d} | [{weights_pattern}]"
    )

mad_values = np.array(
    [result["Best_MAD"] for result in sensitivity_results],
    dtype=float,
)
mad_mean = np.mean(mad_values)
mad_cv = np.std(mad_values) / abs(mad_mean) if mad_mean != 0 else float("nan")
print(f"\nCoefficient of variation for MAD: {mad_cv:.4f}")

if np.isfinite(mad_cv):
    if mad_cv < 0.05:
        print("The observed MAD values vary little across these settings and runs.")
    elif mad_cv < 0.10:
        print("The observed MAD values show moderate variation across these settings and runs.")
    else:
        print("The observed MAD values show substantial variation across these settings and runs.")
else:
    print("The coefficient of variation is undefined because the mean MAD is zero.")

print(
    "This experiment describes variation for the tested settings and stochastic runs; "
    "it is not a guarantee of future forecasting performance."
)

## Interpreting the Optimization Analysis

### What the outputs establish

The comparison table reports rolling-backtest MAD for the simple moving average, the linearly weighted moving average, and the best candidate found by PSO for each tested window size. The reported improvement is relative to the linearly weighted baseline on the same target periods. A positive value means the PSO candidate has a lower MAD on those historical targets; a negative value means it has a higher MAD. Neither outcome establishes which method will forecast future demand more accurately.

The fitness history shows the best MAD encountered as the search proceeds. Because the global best is retained, this history should be non-increasing apart from floating-point representation. The diversity history tracks the average pairwise distance among particle positions: decreasing diversity can indicate that particles are gathering in a smaller region, but it does not by itself show that the region contains a global optimum.

The weight plot uses an oldest-to-newest ordering. Its first bar is the weight on the oldest observation in the moving-average window, and its last bar is the weight on the most recent observation. The next-period forecast is the weighted sum of the latest observations using the best-found weight vector.

### Sensitivity and limitations

The sensitivity experiment changes selected PSO settings and performs stochastic runs. Its MAD coefficient of variation summarizes the dispersion among those reported runs; it does not isolate parameter effects from random initialization and does not measure generalization to unseen demand. The implementation does not set a random seed, so repeated executions can produce different search trajectories and best-found candidates.

The weight search is constrained to nonnegative weights that sum to one. The backtest uses overlapping historical observations, and the same data both guides weight selection and measures MAD. This can make the selected weights fit the example series particularly well. A separate evaluation period would be needed to assess out-of-sample accuracy. PSO is a metaheuristic search method, so convergence of its stopping criterion is not proof that the best possible weights have been found.

### Practical use

This implementation demonstrates how PSO can be applied when WMA weights are treated as continuous decision variables under simplex constraints. In an operational forecasting setting, model choice should consider out-of-sample accuracy, the cost of forecast errors, data quality, and the computational effort required. The example's historical MAD and next-period forecast are outputs of this specific series and run, not general performance guarantees.